# Feature engineering: publishing the assays to Feast

No derived features — the chemistry is the feature set. Engineering here is
making the assays *store objects* the pipeline can retrieve reproducibly.

## Local AWS stack

In [ ]:
%%bash
cat docker-compose.yml

In [ ]:
%%bash
make up-moto

## Build + publish the tables

In [ ]:
%%bash
python -m wine_quality_classification.prepare_data

In [ ]:
%%bash
mkdir -p feature_repo/data && cp data/processed/wine_features.parquet data/processed/wine_labels.parquet feature_repo/data/

In [ ]:
%%bash
cd feature_repo && feast apply

## Point-in-time training frame

In [ ]:
import pandas as pd
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")
entities = pd.read_parquet("data/processed/wine_labels.parquet")
training = store.get_historical_features(
    entity_df=entities,
    features=[
        f"wine_assay_profile:{c}"
        for c in __import__(
            "wine_quality_classification.feature_contract", fromlist=["x"]
        ).FEATURE_COLUMNS
    ]
    + ["wine_cultivar_labels:cultivar"],
).to_df()
training.head()